# Curriculum 03 · Lab 3 — Qdrant: the in-memory cosine store

**Goal:** Index the same corpus into Qdrant's in-memory mode and focus on the
second axis that separates vector stores: **what the score means**. FAISS
(lab 01) and Chroma (lab 02) both report squared-L2 distance (lower =
better); Qdrant reports cosine similarity (higher = better). Same vectors,
same ranking — different numbers.

```
Mode        : path=":memory:" — nothing on disk, forgotten at process exit
Score       : cosine SIMILARITY — HIGHER = more similar (Qdrant's default)
Identity    : cos = 1 - sqL2/2 for unit-norm vectors (proved hit-by-hit)
Persistent  : same API, one-argument change: path="<dir>" instead of ":memory:"
Embedding   : BGE (BAAI/bge-base-en-v1.5, local, CPU)
Data        : rag-mini-wikipedia (first 100 passages, questions 1606/1610)
```

**Why the score matters:** "0.4255" (squared L2) and "0.7873" (cosine) can be
the *same retrieval* — the lab prints both for every hit and shows they obey
`cos = 1 - sqL2/2` to ~4 decimals. That identity is the cross-check that
proves both stores are computing the same similarity and only disagreeing
about how to display it. Never compare scores across stores or embedding
models — compare rankings.

This is the third lab of track 03-vector-databases (see
`.omo/plans/layer1-rag-playbook.md`).


## 0 · Setup — environment, imports & repo paths

**WHAT:** Installs the lab's dependencies (a no-op if already present),
imports pandas plus the repo's vector-store classes and BGE embedder, and
puts the repo-root component library on `sys.path` so this notebook reuses
`src/vectordb/*.py` and `src/embeddings/bge.py` exactly like the lab script.

**WHY:** Everything embeds **locally** with BGE via sentence-transformers —
no API embeddings anywhere. The store classes live in the repo's shared
component library (`src/vectordb/`), not inside the lab, so the exact same code
path runs here, in the `.py`, and in later tracks.

**Paths:** the next cell resolves the **repo root** automatically — it works
whether the kernel launches from the repo root (like the lab script) or from
the notebook's own folder (the Jupyter default) — and `cd`s into it so every
path stays repo-relative.

**WHAT TO EXPECT:** no output from the pip cell (packages already
installed), a silent import from the second. The BGE model is loaded lazily
when the experiment cell first calls it.


In [1]:
# Lab-specific dependencies (already in requirements.txt — the install is a
# no-op safety net for fresh environments):
#   sentence-transformers -> local BGE embeddings
#   faiss-cpu             -> the lab-01 sq-L2 score baseline
#   qdrant-client         -> pinned ==1.13.3 (see requirements.txt: langchain-qdrant
#                            1.1.0 passes init_from to create_collection, which newer
#                            qdrant-client rejects with AssertionError)
#   pandas                -> reads the passages/test.parquet corpus
%pip install sentence-transformers faiss-cpu "qdrant-client==1.13.3" pandas



[notice] A new release of pip is available: 24.2 -> 26.2
[notice] To update, run: python -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import os
import sys
import time
from pathlib import Path

import pandas as pd

# Make the repo-root component library importable. A notebook has no
# ``__file__``, so resolve the repo root by walking up from the kernel's
# working directory — this works whether the kernel launches from the repo
# root (like the lab script) or from the notebook's own folder (Jupyter's
# default) — then cd into it so every repo-relative path behaves exactly
# like the .py.
REPO_ROOT = Path.cwd()
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / "src" / "curriculum").is_dir() and (candidate / "NoteBooks").is_dir():
        REPO_ROOT = candidate
        break
os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT / "src"))

from embeddings.bge import BGEEmbedding  # noqa: E402
from langchain_core.documents import Document  # noqa: E402
from vectordb.faiss import FAISSVectorStore  # noqa: E402
from vectordb.qdrant import QdrantVectorStore  # noqa: E402


## 1 · Configuration — the experiment's knobs

**WHAT:** Same corpus/embedding constants as labs 01–02 plus the
Qdrant-specific ones: `COSINE_TOL` (the `cos = 1 - sqL2/2` identity holds up
to float32 rounding) and `COLLECTION = "lab03"`.

**WHY:** The two-question set is enough because the lab's subject is the
score convention, and the cross-check runs on every single hit.


In [3]:
PASSAGES_PATH = Path("Data/corpus/rag-mini-wikipedia/passages.parquet")
TEST_PATH = Path("Data/corpus/rag-mini-wikipedia/test.parquet")
N_PASSAGES = 100  # deterministic head of the 3200-passage corpus
QUESTION_IDS = [1606, 1610]  # real questions; answers live inside the subset
TOP_K = 3
PREVIEW = 62
BGE_MODEL_NAME = "BAAI/bge-base-en-v1.5"
BGE_DIM = 768
COSINE_TOL = 1e-3  # cos = 1 - sqL2/2 holds up to float32 rounding
COLLECTION = "lab03"


## 2 · Load — corpus + questions + the score-conversion helper

**WHAT:** The same corpus/format helpers as labs 01–02, plus
`expected_cosine` — the exact `cos = 1 - sqL2/2` identity the lab uses as a
cross-check on every hit.

**WHY:** Keeping the identity in one named helper (instead of writing
`1.0 - score / 2.0` inline) makes the gate's assertion readable: "every
Qdrant score matches `expected_cosine(faiss_score)`".


In [4]:
def load_passages(path: Path, n: int) -> tuple[list[str], list[int]]:
    """Return (passage_texts, passage_ids) for the first ``n`` passages."""
    df = pd.read_parquet(path)
    subset = df.head(n)
    return subset["passage"].tolist(), subset.index.tolist()


def load_questions(path: Path, ids: list[int]) -> list[tuple[int, str]]:
    """Return [(question_id, question_text)] for the requested test rows."""
    df = pd.read_parquet(path)
    rows = df.loc[ids]
    return [(int(idx), row["question"]) for idx, row in rows.iterrows()]


def preview(text: str, limit: int = PREVIEW) -> str:
    """Flatten a passage for one-line printing."""
    flat = text.replace("\n", " ")
    return flat[:limit] + ("..." if len(flat) > limit else "")


def expected_cosine(sq_l2: float) -> float:
    """cos(a,b) = 1 - |a-b|^2 / 2, exact for unit-norm vectors."""
    return 1.0 - sq_l2 / 2.0


## 3 · Experiment — embed once, index into FAISS (sq-L2) and Qdrant (cosine)

**WHAT:** `run_experiment` embeds the subset once and feeds the *same*
vectors to a FAISS store (the squared-L2 baseline) and a Qdrant store in
`:memory:` mode. Querying both with the same query vectors yields two score
lists for the same ranking — which the demo lines up hit-by-hit.

**WHY:** The whole lab is one controlled comparison: identical vectors,
identical queries, two stores, two score conventions. Nothing else varies.


In [5]:
def run_experiment() -> dict:
    passage_texts, passage_ids = load_passages(PASSAGES_PATH, N_PASSAGES)
    questions = load_questions(TEST_PATH, QUESTION_IDS)
    chunks = [
        Document(page_content=t, metadata={"id": pid})
        for t, pid in zip(passage_texts, passage_ids)
    ]

    # --- Embed the subset once; BOTH stores index the same vectors ----------
    embedder = BGEEmbedding(model_name=BGE_MODEL_NAME)
    t0 = time.perf_counter()
    passage_vecs = embedder.embed_documents(passage_texts)
    embed_s = time.perf_counter() - t0
    query_vecs = [embedder.embed_query(q) for _, q in questions]

    # --- FAISS (sq-L2, lower = better) — the cross-check baseline -----------
    faiss_store = FAISSVectorStore()
    faiss_store.add(chunks, embeddings=passage_vecs)
    faiss_scored = [faiss_store.query_with_scores(q, top_k=TOP_K) for q in query_vecs]

    # --- Qdrant (cosine, higher = better) — in-memory, nothing on disk ------
    qdrant_store = QdrantVectorStore(
        collection_name=COLLECTION, path=":memory:"
    )
    t0 = time.perf_counter()
    qdrant_store.add(chunks, embeddings=passage_vecs)
    add_s = time.perf_counter() - t0
    qdrant_scored = [qdrant_store.query_with_scores(q, top_k=TOP_K) for q in query_vecs]

    return {
        "passage_texts": passage_texts,
        "passage_ids": passage_ids,
        "questions": questions,
        "embed_s": embed_s,
        "add_s": add_s,
        "faiss_scored": faiss_scored,
        "qdrant_scored": qdrant_scored,
        "dim": len(passage_vecs[0]),
        "indexed": len(passage_vecs),
    }


## 4 · Run — execute the experiment

**WHAT:** Calls `run_experiment()` — one embed, one index build per store,
all queries — and keeps the artifact dict as `exp`.

**WHY:** Everything after this cell (the demo and the verification gate)
reads from this single `exp`, so the printed numbers and the verified
numbers are guaranteed to come from the same run.


In [6]:
exp = run_experiment()


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

## 5 · Demo — read the artifact

**WHAT:** `print_demo` prints the top-3 per question with three columns:
FAISS's squared-L2 score, the cosine value that score *implies*
(`1 - sqL2/2`), and the cosine Qdrant actually reports. The `SAME` labels
confirm identical rankings.

**WHY:** Read a row like a cross-check: `faiss 0.4255 | 1-sqL2/2 0.7873 |
qdrant 0.7873` — two stores, three numbers, one retrieval. When the two
cosine columns agree to ~4 decimals, both stores are provably computing the
same similarity.


In [7]:
def print_demo(exp: dict) -> None:
    print("=" * 66)
    print("Lab 03 — Qdrant: the in-memory cosine store")
    print(f"{BGE_MODEL_NAME} | cosine distance | path=:memory: (nothing on disk)")
    print("=" * 66)

    print(f"\n[1] Corpus + embedding:")
    print(f"    {exp['indexed']} passages, dim {exp['dim']}, embedded in {exp['embed_s']:.2f}s")
    print(f"    same vectors indexed into FAISS (sq-L2) AND Qdrant (cosine)")

    print(f"\n[2] Qdrant index build (in-memory):")
    print(f"    {exp['indexed']} passages added in {exp['add_s']:.3f}s")
    print("    no persist_dir: path=\":memory:\" writes zero bytes and forgets")
    print("    everything at process exit (persistent mode = path='<dir>' instead)")

    print(f"\n[3] Top-{TOP_K} per question — sq-L2 (FAISS) vs cosine (Qdrant):")
    print("    col '1 - sqL2/2' is the cosine value FAISS's score implies;")
    print("    col 'qdrant' is what Qdrant actually reports. They should match.")
    for i, (qid, qtext) in enumerate(exp["questions"]):
        print(f'\n    Q[{qid}] "{qtext}"')
        for (fdoc, fscore), (cdoc, cscore) in zip(
            exp["faiss_scored"][i], exp["qdrant_scored"][i]
        ):
            conv = expected_cosine(fscore)
            match = "SAME" if fdoc.metadata["id"] == cdoc.metadata["id"] else "DIFF"
            print(f"      faiss {fscore:8.4f} | 1-sqL2/2 {conv:8.4f} | qdrant {cscore:8.4f} "
                  f"| [passage {cdoc.metadata['id']}] {preview(cdoc.page_content)}  {match}")

    print("\n[4] Takeaway")
    print("    The ranking is identical to labs 01/02 — the store never changes")
    print("    WHAT ranks first, only the score scale and direction. For unit-")
    print("    norm vectors cos = 1 - sqL2/2, so '0.4255 sq-L2' and '0.7873")
    print("    cosine' are the same retrieval. Never compare scores across")
    print("    stores (or across embedding models); compare rankings. And:")
    print("    :memory: is a scratchpad — the same one-line API turns it into")
    print("    a persistent store when you pass a directory path.")


In [8]:
print_demo(exp)


Lab 03 — Qdrant: the in-memory cosine store
BAAI/bge-base-en-v1.5 | cosine distance | path=:memory: (nothing on disk)

[1] Corpus + embedding:
    100 passages, dim 768, embedded in 15.87s
    same vectors indexed into FAISS (sq-L2) AND Qdrant (cosine)

[2] Qdrant index build (in-memory):
    100 passages added in 0.818s
    no persist_dir: path=":memory:" writes zero bytes and forgets
    everything at process exit (persistent mode = path='<dir>' instead)

[3] Top-3 per question — sq-L2 (FAISS) vs cosine (Qdrant):
    col '1 - sqL2/2' is the cosine value FAISS's score implies;
    col 'qdrant' is what Qdrant actually reports. They should match.

    Q[1606] "Is Uruguay's capital Montevideo?"
      faiss   0.2650 | 1-sqL2/2   0.8675 | qdrant   0.8675 | [passage 36] Montevideo, Uruguay's capital.  SAME
      faiss   0.4931 | 1-sqL2/2   0.7535 | qdrant   0.7535 | [passage 15] Uruguay's capital, Montevideo, was founded by the Spanish in t...  SAME
      faiss   0.5141 | 1-sqL2/2   0.7430 

## 6 · Verification gate — the same checks the .py runs

**WHAT:** Runs the exact `verify_gate`: dimension/count, Qdrant ranking
identical to FAISS, cosine scores descending with rank, every Qdrant score
matching `1 - sqL2/2` from the FAISS score, and both content checks.

**WHY:** `python 03-qdrant-in-memory.py --verify` must print 7/7 PASS; this
cell proves the notebook reproduces the verified `.py` exactly.


In [9]:
def verify_gate(exp: dict) -> int:
    checks: list[tuple[str, bool]] = []

    checks.append(("embedding dimension is 768 (BGE base)", exp["dim"] == BGE_DIM))
    checks.append((f"exactly {N_PASSAGES} passages indexed", exp["indexed"] == N_PASSAGES))

    # Same embeddings => same ranking, whatever the store.
    all_same_rank = True
    for fhits, qhits in zip(exp["faiss_scored"], exp["qdrant_scored"]):
        for (fdoc, _), (qdoc, _) in zip(fhits, qhits):
            all_same_rank &= fdoc.metadata["id"] == qdoc.metadata["id"]
    checks.append(("Qdrant ranks the same passages as FAISS, in the same order", all_same_rank))

    # The signature assertion: Qdrant's cosine score equals 1 - sqL2/2 for
    # every hit, i.e. both stores are computing the same similarity.
    cos_consistent = True
    descending = True
    for fhits, qhits in zip(exp["faiss_scored"], exp["qdrant_scored"]):
        q_scores = [s for _, s in qhits]
        descending &= q_scores == sorted(q_scores, reverse=True)
        for (_, fscore), (_, cscore) in zip(fhits, qhits):
            cos_consistent &= abs(expected_cosine(fscore) - cscore) < COSINE_TOL
    checks.append(("cosine scores descend with rank (higher = more similar)", descending))
    checks.append(("every Qdrant score matches 1 - sqL2/2 from the FAISS score", cos_consistent))

    # Content checks (same as labs 01/02).
    q1610_top = exp["qdrant_scored"][1][0][0].page_content.lower()
    checks.append(("Q1610 top-1 names the Spanish founder of Montevideo", "spanish" in q1610_top))
    q1606_top = exp["qdrant_scored"][0][0][0].page_content.lower()
    checks.append(("Q1606 top-1 mentions Montevideo", "montevideo" in q1606_top))

    print("verification gate:")
    for label, ok in checks:
        print(f"  [{'PASS' if ok else 'FAIL'}] {label}")
    return 0 if all(ok for _, ok in checks) else 1


if __name__ == "__main__":
    exp = run_experiment()
    if "--verify" in sys.argv:
        sys.exit(verify_gate(exp))
    print_demo(exp)


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Lab 03 — Qdrant: the in-memory cosine store
BAAI/bge-base-en-v1.5 | cosine distance | path=:memory: (nothing on disk)

[1] Corpus + embedding:
    100 passages, dim 768, embedded in 6.47s
    same vectors indexed into FAISS (sq-L2) AND Qdrant (cosine)

[2] Qdrant index build (in-memory):
    100 passages added in 0.091s
    no persist_dir: path=":memory:" writes zero bytes and forgets
    everything at process exit (persistent mode = path='<dir>' instead)

[3] Top-3 per question — sq-L2 (FAISS) vs cosine (Qdrant):
    col '1 - sqL2/2' is the cosine value FAISS's score implies;
    col 'qdrant' is what Qdrant actually reports. They should match.

    Q[1606] "Is Uruguay's capital Montevideo?"
      faiss   0.2650 | 1-sqL2/2   0.8675 | qdrant   0.8675 | [passage 36] Montevideo, Uruguay's capital.  SAME
      faiss   0.4931 | 1-sqL2/2   0.7535 | qdrant   0.7535 | [passage 15] Uruguay's capital, Montevideo, was founded by the Spanish in t...  SAME
      faiss   0.5141 | 1-sqL2/2   0.7430 |

In [10]:
verify_gate(exp)


verification gate:
  [PASS] embedding dimension is 768 (BGE base)
  [PASS] exactly 100 passages indexed
  [PASS] Qdrant ranks the same passages as FAISS, in the same order
  [PASS] cosine scores descend with rank (higher = more similar)
  [PASS] every Qdrant score matches 1 - sqL2/2 from the FAISS score
  [PASS] Q1610 top-1 names the Spanish founder of Montevideo
  [PASS] Q1606 top-1 mentions Montevideo


0